# 00 · Project setup and data check

Run this once at the start, and again whenever a data file changes. It

1. creates the project folders and `config.json` on Drive,
2. checks that both raw data files are where `config.json` says, and records their size and MD5 hash,
3. fixes the train/validation/test split for every seed and saves it in `data/splits/`, so every later notebook uses exactly the same rows.

No model is trained here. Takes a minute or two (most of it is hashing the PaySim file).

**Before running:** upload `rareguard_lib.py` into `RAREGuard_v6/code/` on Drive.

## Folders and config

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

lib = os.path.join(ROOT, "code", "rareguard_lib.py")
if not os.path.exists(lib):
    os.makedirs(os.path.dirname(lib), exist_ok=True)
    raise FileNotFoundError(f"Put rareguard_lib.py in {os.path.dirname(lib)} and run this cell again.")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.init_project(ROOT)
env = rg.show_environment()
print("\nData paths in config.json:")
for k, v in cfg["data"].items():
    print(f"  {k:<11} {v}")

If either path is wrong, open `RAREGuard_v6/config.json` in Drive (double-click, then *Open with* > Text Editor), fix it, and rerun the cell above. All other settings in that file are the ones used in the v5 notebooks; leave them alone unless you mean to change the experiment.

## Raw data files

In [ ]:
import numpy as np, pandas as pd

run = rg.Run(cfg, "E0_data", "00_setup_and_data.ipynb", "Data files, dataset summary and fixed splits")

checks = []
for name, path in cfg["data"].items():
    ok = os.path.exists(path)
    checks.append({"File": name, "Path": path, "Exists": ok,
                   "Size_MB": round(os.path.getsize(path) / 2**20, 1) if ok else None,
                   "MD5": rg.file_md5(path) if ok else None})
checks = pd.DataFrame(checks)
display(checks)
assert checks.Exists.all(), "A data file is missing - fix the path in config.json"
run.save_csv(checks, "data_files.csv")

The MD5 hash identifies the exact file version. If you ever download the data again and the hash changes, the saved splits no longer apply and `rg.get_split` will refuse to load them.

## ULB credit card data: summary and splits

In [ ]:
X, y, feature_names = rg.load_ulb(cfg)
print(f"{len(y):,} transactions, {int(y.sum())} frauds ({100 * y.mean():.4f}%), {len(feature_names)} features")
if len(y) != 284807 or int(y.sum()) != 492:
    print("Note: this is not the usual 284,807 / 492 version of the dataset.")

ulb_splits = {seed: rg.get_split(cfg, "ulb", y, seed) for seed in cfg["seeds"]}
ulb_summary = rg.split_summary(y, ulb_splits)
display(ulb_summary)
run.save_csv(ulb_summary, "ulb_splits.csv")

## PaySim: fixed 300,000-row sample and splits

In [ ]:
pay = rg.load_paysim_sample(cfg)
y_pay = pay["isFraud"].to_numpy(dtype=np.int64)
print(f"PaySim sample: {len(pay):,} rows, {int(y_pay.sum())} frauds ({100 * y_pay.mean():.4f}%)")

# Row numbers of the sampled transactions in the original file, so the sample can be rebuilt or checked later
np.save(rg.proj(cfg, "data", "splits", "paysim_sample_rows.npy"), pay["source_row"].to_numpy())

pay_splits = {seed: rg.get_split(cfg, "paysim", y_pay, seed) for seed in cfg["seeds"]}
pay_summary = rg.split_summary(y_pay, pay_splits)
display(pay_summary)
run.save_csv(pay_summary, "paysim_splits.csv")

by_type = pay.groupby("type").agg(Transactions=("isFraud", "size"), Fraud=("isFraud", "sum"))
by_type["Fraud_Rate"] = by_type.Fraud / by_type.Transactions
display(by_type)
run.save_csv(by_type.reset_index(), "paysim_fraud_by_type.csv")

In [ ]:
run.finish()